# 🚀 Distill a tool-use agent using Knowledge Distillation on Amazon SageMaker AI

Sequence-level knowledge distillation (SeqKD) of a shopping **tool-use agent** into `Qwen/Qwen3.5-4B`,
trained with the **serverless `SFTTrainer`** from the SageMaker Python SDK v3.

SeqKD needs no distillation-specific algorithm: a larger teacher generates tool-use trajectories, and the
student is trained on them with ordinary SFT (loss masked to the completion). The teacher trajectories
persist as a dataset, so you can reuse them to train other students.

**What you will learn**

1. Encode teacher ShoppingBench trajectories into `{prompt, completion}` pairs with the student's exact
   non-thinking chat template.
2. Run SeqKD (SFT on those pairs) as a SageMaker serverless training job.
3. Deploy the customized student and evaluate task success (ASR) + IFEval regression.

**Why the serverless `SFTTrainer`:** SageMaker runs a managed recipe, so you supply a dataset and
hyperparameters and nothing else. You don't choose an instance type, build a training image, or write a
training script — and, specifically for this model, you don't hand-patch kernels. Qwen3.5-4B is a
linear-attention/conv1d **hybrid** whose linear-attn path needs `flash-linear-attention` force-installed
against a conflicting `triton` pin, plus `sdpa` attention to avoid a CUDA illegal-memory-access on the
managed driver. The recipe handles that. The job also emits an **already-merged, registered Model
Package**, which you deploy and evaluate directly — no adapter-merge step.

**Results with this notebook** (250-case ShoppingBench test set):

| Metric | Base Qwen3.5-4B | Distilled student | Δ |
|---|---|---|---|
| Tool-use **ASR** (task success rate) | 0.432 | **0.636** | **+20.4 pts** |
| **IFEval** (prompt-level strict, non-thinking) | 0.826 | 0.782 | −4.4 pts |

Agent skill rises sharply while general instruction-following holds — the trade you want from SeqKD.

## 00. Setup

The serverless path lives in SDK **v3** namespaces: `sagemaker.train.sft_trainer.SFTTrainer`,
`sagemaker.ai_registry.dataset.DataSet`, and `sagemaker.core.resources`.

In [ ]:
import os, json, boto3
from sagemaker.core.helper.session_helper import get_execution_role

REGION = boto3.Session().region_name or "us-west-2"
os.environ["AWS_DEFAULT_REGION"] = os.environ["AWS_REGION"] = REGION

# Needed for native MLflow metrics while the trainer waits:
os.environ["SAGEMAKER_MLFLOW_CUSTOM_ENDPOINT"] = f"https://mlflow.sagemaker.{REGION}.app.aws"

try:
    ROLE = get_execution_role()
    print("Execution role:", ROLE)
except Exception:
    ACCOUNT = boto3.client("sts").get_caller_identity()["Account"]
    ROLE = f"arn:aws:iam::{ACCOUNT}:role/Dev"        # <- replace with your execution role

BUCKET  = "<name>"                                   # <- replace with your bucket
PREFIX  = "<name>"
S3_ROOT = f"s3://{BUCKET}/{PREFIX}"
print(REGION, ROLE, S3_ROOT)

## 01. Data Prep — encode teacher trajectories → `{prompt, completion}`

One training example per assistant step; loss falls only on the completion. We render with the **student's own chat template** (`apply_chat_template`, `enable_thinking=False`, `tools=`) via a prefix-diff so `prompt + completion` is byte-identical to inference (non-thinking: prompt ends with a closed-empty `<think></think>`; tool calls render in Qwen-native format). TRL auto-detects the prompt/completion format and masks the prompt. Swap `REPO`/`sft_file` for your own teacher trajectories.

In [ ]:
import json, random
from huggingface_hub import list_repo_files, hf_hub_download
from transformers import AutoTokenizer
HF_STUDENT="Qwen/Qwen3.5-4B"
REPO = "oro-ai/sn15-shoppingbench-sft-15k"   # example teacher trajectories
jsonls = [f for f in list_repo_files(REPO, repo_type="dataset") if f.endswith(".jsonl")]
sft_file = next(f for f in jsonls if "metadata" not in f.lower() and "eval" not in f.lower())
raw = [json.loads(l) for l in open(hf_hub_download(REPO, sft_file, repo_type="dataset")) if l.strip()]
raw = [r for r in raw if r.get("messages")]
print(len(raw), "trajectories")

tok = AutoTokenizer.from_pretrained(HF_STUDENT, trust_remote_code=True)
SYSTEM = ("You are a shopping assistant. Use the available tools to satisfy the request: "
          "search the catalog, inspect candidates, verify the constraints, then recommend.")

def _render(msgs, tools, add_gen):
    return tok.apply_chat_template(msgs, tools=tools, tokenize=False,
                                   add_generation_prompt=add_gen, enable_thinking=False)

def to_rows(traj):
    tools = traj.get("tools")
    convo = [{"role": "system", "content": SYSTEM}] + [m for m in traj["messages"] if m.get("role") != "system"]
    rows = []
    for i, m in enumerate(convo):
        if m.get("role") != "assistant":
            continue
        prompt = _render(convo[:i],     tools, add_gen=True)
        full   = _render(convo[:i + 1], tools, add_gen=False)
        assert full.startswith(prompt), "chat-template prefix mismatch"
        completion = full[len(prompt):]
        if completion.strip():
            rows.append({"prompt": prompt, "completion": completion})
    return rows

examples = [r for traj in raw for r in to_rows(traj)]
random.seed(42); random.shuffle(examples)
val, train = examples[:1000], examples[1000:]
print(f"{len(raw)} trajectories -> {len(examples)} SFT examples  (train {len(train)} / val {len(val)})")

def write_jsonl(rows, p):
    with open(p, "w") as f:
        for r in rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")
write_jsonl(train, "seqkd_train.jsonl"); write_jsonl(val, "seqkd_val.jsonl")

#### Upload to S3, then register the dataset in the SageMaker AI Registry

The serverless `SFTTrainer` consumes a **registered dataset ARN**, not a raw S3 path. `DataSet.create`
wraps a JSONL file in a versioned registry entry. Each line must carry `prompt` / `completion` fields —
exactly what the cell above wrote.

In [ ]:
from sagemaker.ai_registry.dataset import DataSet

s3 = boto3.client("s3", region_name=REGION)
s3.upload_file("seqkd_train.jsonl", BUCKET, f"{PREFIX}/data/seqkd_train.jsonl")
s3.upload_file("seqkd_val.jsonl",   BUCKET, f"{PREFIX}/data/seqkd_val.jsonl")
TRAIN_S3 = f"{S3_ROOT}/data/seqkd_train.jsonl"
VAL_S3   = f"{S3_ROOT}/data/seqkd_val.jsonl"

train_dataset = DataSet.create(name="seqkd-shopping-train", source=TRAIN_S3)
val_dataset   = DataSet.create(name="seqkd-shopping-val",   source=VAL_S3)
print("train:", train_dataset.arn)
print("val:  ", val_dataset.arn)

## 02. Fine-tune (SeqKD) with serverless `SFTTrainer`

Distillation here *is* ordinary SFT on the teacher trajectories. You give the trainer a base model id, a
training type, the two registered datasets, and an output path.

`model_package_group` is **required** when you pass a base model id — it's where the output Model Package
lands. `accept_eula=True` is required for gated base models.

In [ ]:
from sagemaker.train.sft_trainer import SFTTrainer
from sagemaker.train.common import TrainingType
from sagemaker.core.resources import ModelPackageGroup

MPG_NAME = "seqkd-qwen35-4b-shop"
try:
    model_package_group = ModelPackageGroup.create(model_package_group_name=MPG_NAME)
except Exception as e:
    print("reusing existing group:", type(e).__name__)
    model_package_group = MPG_NAME          # an existing name or ARN is also accepted

# To find out which models are supported by serverless training:
# SFTTrainer.list_supported_models()

sft_trainer = SFTTrainer(
    model="huggingface-vlm-qwen3-5-4b",     # the Qwen3.5-4B student
    training_type=TrainingType.FULL,        # TrainingType.FULL for FFT -- read the note below first
    model_package_group=model_package_group,
    mlflow_experiment_name="seqkd-shopping-exp",
    mlflow_run_name="seqkd-shopping",
    training_dataset=train_dataset.arn,
    validation_dataset=val_dataset.arn,
    s3_output_path=f"{S3_ROOT}/output/",
    accept_eula=True,
)

### Inspect, then override the hyperparameters if needed

Print the recipe defaults first and change only what you need.

In [ ]:
print("Recipe defaults:")
pprint(sft_trainer.hyperparameters.to_dict())

In [ ]:
sft_trainer.hyperparameters.dataset_max_len       = 16384
sft_trainer.hyperparameters.max_epochs            = 2
sft_trainer.hyperparameters.global_batch_size     = 64
sft_trainer.hyperparameters.learning_rate         = 0.00002
sft_trainer.hyperparameters.train_val_split_ratio = 0.9
sft_trainer.hyperparameters.lr_warmup_steps_ratio = 0.03
pprint(sft_trainer.hyperparameters.to_dict())

### Dry-run, then train

`dry_run=True` validates IAM permissions, dataset ARNs, hyperparameters, and infrastructure without
submitting a job — cheap insurance before a multi-hour run.

In [ ]:
sft_trainer.train(dry_run=True)
print("Dry run passed - configuration is valid.")

In [ ]:
training_job = sft_trainer.train(wait=True)

# To submit now and monitor later instead:
#   training_job = sft_trainer.train(wait=False)
#   sft_trainer.stream_logs()

print(training_job.training_job_name, training_job.training_job_status)

> **Hyperparameter notes, from the runs behind the table at the top.**
>
> - **`dataset_max_len = 16384`.** Multi-turn ReAct trajectories accumulate tool output. At 4096 roughly
>   19% of this dataset's trajectories get truncated mid-rollout, and ASR pays for it.
> - **`learning_rate = 5e-05` at `global_batch_size = 64`.** If the recipe pins you to a large global
>   batch, scale the LR to it rather than fighting it. Scale by `sqrt` if you change the batch size.
> - **FFT arm (`TrainingType.FULL`):** drop LR to **`2e-05`** and set `weight_decay = 0.0`. FFT reaches a
>   higher ASR ceiling (0.736 measured) but on shop-only data it catastrophically forgets — IFEval
>   collapsed to 0.155. Mixing roughly **30% general instruction data** back into the training file
>   recovered it (IFEval 0.702). Don't run pure FFT on agent-only data unless you accept that trade.
>   Data mixing matters more here than any LR choice.
> - LoRA at `r=16, alpha=128` was sufficient; raising the rank was not what moved ASR. Note that LoRA
>   scaling is `alpha / rank`, so going `r=16 → r=64` at a fixed `alpha=128` *weakens* the update 4× — it
>   is not a pure capacity increase. To isolate capacity at `r=64`, raise `alpha` to 512.
> - **Step count scales with your dataset.** At `train_val_split_ratio = 0.9` this file yields ~51.5k
>   training rows, so `global_batch_size = 64` is ~805 optimizer steps per epoch (~1,610 for 2 epochs).
>   A warmup ratio of `0.03`–`0.05` is appropriate at that length; a few-hundred-example run would need
>   proportionally less.

## 03. Collect the output Model Package

**There is no merge step here, and that is the point.** A self-managed LoRA job writes an *adapter* that
you then have to download, merge into the base weights, re-add the VLM processor configs
(`preprocessor_config.json`, `processor_config.json`), and re-upload. The serverless job registers a
ready-to-deploy **Model Package** instead; the completed job's `output_model_package_arn` points at it.

In [ ]:
job = TrainingJob.get(training_job_name=training_job.training_job_name)
MODEL_PKG_ARN = job.output_model_package_arn
print("Output Model Package ARN:", MODEL_PKG_ARN)

model_package = ModelPackage.get(model_package_name=MODEL_PKG_ARN)
print("status:", model_package.model_package_status)

# To find the package later without the job object:
#   aws sagemaker list-model-packages --model-package-group-name seqkd-qwen35-4b-shop \
#     --sort-by CreationTime --sort-order Descending \
#     --query 'ModelPackageSummaryList[0].ModelPackageArn'

## 04. Evaluate

| Axis | Compute | How |
|---|---|---|
| **IFEval + tool-use ASR** | a **real-time endpoint** (GPU) + a CPU-light driver here | deploy once, drive it through a LiteLLM shim |

Generation runs on the endpoint's GPU; the rollout loop and scoring run in this notebook kernel. The
companion **`evaluate-tool-call-accuracy.ipynb`** does the full base-vs-student comparison and chart.

> **Measurement trap: IFEval must be served thinking-OFF.** Qwen3.5 defaults to thinking ON, and its chat
> template then opens a reasoning block — so IFEval scores the *reasoning trace* against the prompt's
> format constraints ("≤100 words", "wrap the response in quotes", "highlight 2 sections") and they fail by
> construction. Same weights, same harness: **0.782 non-thinking vs 0.264 thinking-ON.** The 0.264 is an
> artifact of the serving config, not a property of the model. An endpoint behind `/invocations` can't force
> `enable_thinking=false` through the bridge, so the trustworthy IFEval number comes from a standalone vLLM
> with a non-thinking chat template — see `scripts/eval_lmeval.sh`.

> **Self-contained alternative (no endpoint):** `scripts/eval_lmeval.sh <MODEL_DIR> <IFACE> <GPU> <TAG>`
> runs non-thinking IFEval on a single GPU box, and `scripts/shopping_asr_eval.sh` runs ASR.

### 04a. Deploy the student to a real-time endpoint (`ModelBuilder`)

Deploy straight from the Model Package produced in step 03 — no S3 model artifact plumbing. Name the LMI
image and `model_server=DJL_SERVING` **explicitly**: that image bundles **vLLM 0.20.1**, which already has
native kernels for the Qwen3.5 linear-attn/conv1d hybrid, so **no custom inference image is needed**. The
`triton` / `flash-linear-attention` workarounds are training-side only.

In [ ]:
from sagemaker.serve import ModelBuilder, ModelServer

ENDPOINT_NAME = "seqkd-qwen35-4b-shop"
INSTANCE_TYPE = "ml.g5.2xlarge"          # ml.p4de.24xlarge / ml.p4d.24xlarge also validated
LMI_IMAGE = f"763104351884.dkr.ecr.{REGION}.amazonaws.com/djl-inference:0.36.0-lmi25.0.0-cu130"  # vLLM 0.20.1

mb = ModelBuilder(
    model=MODEL_PKG_ARN,                 # the registered output of the serverless job
    role_arn=ROLE,
    image_uri=LMI_IMAGE,
    model_server=ModelServer.DJL_SERVING,
    instance_type=INSTANCE_TYPE,
    env_vars={"OPTION_ROLLING_BATCH": "vllm", "OPTION_TRUST_REMOTE_CODE": "true",
              "OPTION_TENSOR_PARALLEL_DEGREE": "1", "OPTION_MAX_MODEL_LEN": "8192"},
)
mb.model_name = ENDPOINT_NAME
mb.build()
mb.deploy(endpoint_name=ENDPOINT_NAME, initial_instance_count=1, instance_type=INSTANCE_TYPE)
print("endpoint:", ENDPOINT_NAME)

# Capacity note: g5/g6 2xlarge often return InsufficientInstanceCapacity. A Failed endpoint must be
# deleted before you retry the same name, and SageMaker won't delete one stuck in Creating until it settles.

### 04b. IFEval + tool-use ASR — notebook driver → the endpoint

These run in **this notebook kernel** (CPU-light — orchestration + scoring only) and send requests to the endpoint, whose GPU does the generation. SageMaker endpoints are invoked via `InvokeEndpoint`, not an OpenAI URL, so **LiteLLM** bridges the gap — either call `litellm.completion(model=f"sagemaker/{ENDPOINT_NAME}", ...)` directly, or run `litellm --model sagemaker/<endpoint>` as a proxy that exposes an OpenAI `base_url` for `lm_eval` / the rollout to point at.

**Thinking-off (important):** on a managed endpoint you can't pass `--chat-template nonthink_template.jinja` like the self-hosted path, so force non-thinking **per request** with `extra_body={"enable_thinking": false}` (mirrors the rollout config). IFEval floors (~28%) if thinking is left on.

**ASR** reuses the same ReAct rollout as `scripts/shopping_asr_eval.sh` / `run_rollout.py`, only with `base_url` pointed at the LiteLLM shim instead of a local vLLM. It still needs the **ShoppingBench repo + search server (:5631) + `JAVA_HOME`** for tool results and orm.py scoring.

In [ ]:
# --- VALIDATED 2026-10-01 against a live p4de endpoint (then torn down) ---
# IFEval over the endpoint works via a LiteLLM proxy + lm-eval, with these exact flags:
#   litellm --model sagemaker/<ENDPOINT_NAME> --port 4000 --drop_params      # --drop_params: sagemaker
#     provider rejects lm-eval's `seed` param (400) without it
#   lm_eval --model local-chat-completions \
#     --model_args base_url=http://localhost:4000/v1/chat/completions,model=sagemaker/<ENDPOINT_NAME>,\
#                  num_concurrent=4,tokenizer_backend=huggingface,tokenizer=Qwen/Qwen3.5-4B,eos_string=<|im_end|> \
#     --tasks ifeval --apply_chat_template --gen_kwargs max_tokens=1280
# CAVEAT (important): the DJL/LMI + LiteLLM bridge does NOT propagate enable_thinking=false (and --drop_params
# strips it), so the endpoint serves THINKING-ON -> IFEval is floored (sampled: prompt-strict ~0.18, the
# <think> block eats the token budget). To get the real non-thinking 0.826 you must either serve with the
# non-thinking template (pod: `vllm serve --chat-template nonthink_template.jinja`) or deploy with
# model_server=ModelServer.VLLM (vLLM-native OpenAI server, which honors chat_template_kwargs).

# Invoke schema: this endpoint serves the TGI `{"inputs":..., "parameters":...}` schema (returns {"generated_text"}).
# The OpenAI `{"messages":...}` schema returns 424 unless the chat route is enabled; LiteLLM's `sagemaker/` provider uses the TGI schema under the hood.
# --- LiteLLM proxy: expose the endpoint as an OpenAI base_url ---
#   pip install "litellm[proxy]"
#   litellm --model sagemaker/{ENDPOINT_NAME} --port 4000 &        # -> http://localhost:4000
#
# IFEval (non-thinking, generative) — point lm_eval at the proxy:
#   lm_eval --model local-chat-completions \
#     --model_args base_url=http://localhost:4000/v1/chat/completions,model=sagemaker/{ENDPOINT_NAME},num_concurrent=8 \
#     --apply_chat_template --gen_kwargs enable_thinking=false --tasks ifeval
#   # reference: prompt-strict 0.826 (base) -> 0.782 (student)

# ASR — drive the ShoppingBench ReAct loop against the endpoint (thinking OFF), score with orm.py:
import litellm
def endpoint_chat(messages, tools=None):
    return litellm.completion(model=f"sagemaker/{ENDPOINT_NAME}", messages=messages, tools=tools,
                              temperature=0, extra_body={"enable_thinking": False})
# for ex in eval_dataset: run the tool loop with endpoint_chat() (search server for tool results),
#     then score the final recommendation with orm.py (success rate = rule>=1)
print("ASR reference: base 43.2 -> 62.4 (LoRA r64)")

### 04c. Tear down the endpoint

Delete it when done — a real-time endpoint bills for its instance while it's up.

In [ ]:
# deploy() returns an Endpoint object (no .delete_endpoint), so delete by name via the sagemaker
# client. ALWAYS tear down - an idle endpoint keeps billing (p4de/g5 are not cheap).
import boto3
sm = boto3.client("sagemaker", region_name=REGION)
sm.delete_endpoint(EndpointName=ENDPOINT_NAME)
sm.delete_endpoint_config(EndpointConfigName=ENDPOINT_NAME)
sm.delete_model(ModelName=ENDPOINT_NAME)
print("deleted endpoint + config + model:", ENDPOINT_NAME)